In [ ]:
#r "./../../../public/src/L4-application/BoSSSpad/bin/Release/net8.0/BoSSSpad.dll"
using System;
using System.Threading;
using System.Diagnostics;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Solution;
using BoSSS.Application.XNSE_Solver;
using BoSSS.Application.BoSSSpad;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Control;
using BoSSS.Solution.XNSECommon;
using BoSSS.Solution.NSECommon;
using BoSSS.Solution.LoadBalancing;
using BoSSS.Solution.LevelSetTools;
using BoSSS.Solution.XdgTimestepping;
using BoSSS.Solution.Utils;
using System.IO;
using BoSSS.Application.XNSE_Solver.PhysicalBasedTestcases;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
using MomentFittingVariants = BoSSS.Foundation.XDG.XQuadFactoryHelper.MomentFittingVariants;
Init();

In [ ]:
//defaults
string dbname = System.Environment.GetEnvironmentVariable("DATABASE_NAME");
string buildname = System.Environment.GetEnvironmentVariable("JOB_NAME");
buildname = String.IsNullOrEmpty(buildname)? "CollidingSpheres3D" : buildname; //Popcorn2D  VanishingSphere2D
dbname = String.IsNullOrEmpty(dbname)? "condStudy" : dbname; 
string table_name = String.Concat(buildname, "_", dbname);

In [ ]:
using BoSSS.Solution.Gnuplot;


In [ ]:
//ExecutionQueues
var myBatch = GetDefaultQueue();
string WFlowName = table_name;
BoSSS.Application.BoSSSpad.BoSSSshell.WorkflowMgm.Init(WFlowName,myBatch);
BoSSS.Application.BoSSSpad.BoSSSshell.WorkflowMgm.SetNameBasedSessionJobControlCorrelation();
var myDB = BoSSS.Application.BoSSSpad.BoSSSshell.WorkflowMgm.DefaultDatabase; 
myDB

In [ ]:
myDB.Sessions.Where(s => s.SuccessfulTermination == true)

In [ ]:
BoSSSshell.WorkflowMgm.condReader.Update();

In [ ]:
var Tab = BoSSSshell.WorkflowMgm.SessionTable;

In [ ]:
var Tab2 = Tab.ExtractColumns( "AgglomerationThreshold",  "DGdegree:Velocity*","Grid:hMin","Grid:NoOfCells","#timestep", "MassMtxCondNo", "TotCondNo-Vars0.1.2");
Tab2.SaveToFile(table_name + $"_condSummary.json");
Tab2.Print()

In [ ]:
using BoSSS.Solution.Gnuplot;
using System.Linq;

// arrange line color and types
var allColors = Enum.GetValues(typeof(LineColors)).Cast<LineColors>().ToArray();
var allPointTypes = Enum.GetValues(typeof(PointTypes)).Cast<PointTypes>().ToArray();
PointTypes[] myPointTypes = new PointTypes[]{ PointTypes.Diamond, PointTypes.Box,  PointTypes.LowerTriangle,PointTypes.OpenLowerTriangle };


Condition numbers for mass matrix

In [ ]:
string xVarName = "#timestep";
string yVarName = "MassMtxCondNo";
int dgDegree = 1;

// Filter rows depending on selection
var Tab3 = Tab2.ExtractRows((iRow,RowEntries)=> 
                //Convert.ToDouble(RowEntries["AgglomerationThreshold"]) != 0 &&
                Convert.ToDouble(RowEntries["DGdegree:Velocity*"]) == dgDegree
            );

// Sort in ascending order of AgglomerationThreshold
System.Data.DataView view = Tab3.DefaultView;
view.Sort = "AgglomerationThreshold ASC"; 
Tab3 = view.ToTable();

Tab3.Print()


In [ ]:

Plot2Ddata plot = new Plot2Ddata();
    int i = 0;
plot.dataGroups = new Plot2Ddata.XYvalues[Tab3.Rows.Count];

foreach (System.Data.DataRow row in Tab3.Rows){
    //row.
    //*Console.WriteLine(row.Summary());
    double aggThre = (double)row.ItemArray[Tab2.Columns.IndexOf("AgglomerationThreshold")];
    var xValue = (List<double>)row.ItemArray[Tab2.Columns.IndexOf(xVarName)];
    var yValue = (List<double>)row.ItemArray[Tab2.Columns.IndexOf(yVarName)];
    Plot2Ddata.XYvalues xyGroup = new Plot2Ddata.XYvalues("AggThreshold="+aggThre, xValue, yValue);
    plot.dataGroups[i] = xyGroup;
    //ArrayTools.AddToArray(xyGroup, ref plot.dataGroups);
            var dataGroup = plot.dataGroups[i];
            dataGroup.Format.LineColor = allColors[i % allColors.Length];
            dataGroup.Format.PointType = myPointTypes[i % myPointTypes.Length];
            i++;
}

In [ ]:
plot.PlotNow()

In [ ]:
plot.SaveToGIF(yVarName + "_k" + dgDegree + "_2D.png");
plot.SavePgfplotsFile(yVarName + "_k" + dgDegree + "_2D.tex");

Condition numbers for operator matrix

In [ ]:
string xVarName = "#timestep";
string yVarName = "TotCondNo-Vars0.1.2";

// Filter rows depending on selection
var Tab3 = Tab2.ExtractRows((iRow,RowEntries)=> 
                //Convert.ToDouble(RowEntries["AgglomerationThreshold"]) != 0 &&
                Convert.ToDouble(RowEntries["DGdegree:Velocity*"]) == dgDegree
            );

// Sort in ascending order of AgglomerationThreshold
System.Data.DataView view = Tab3.DefaultView;
view.Sort = "AgglomerationThreshold ASC"; 
Tab3 = view.ToTable();


In [ ]:

Plot2Ddata plot = new Plot2Ddata();
    int i = 0;
plot.dataGroups = new Plot2Ddata.XYvalues[Tab3.Rows.Count];
foreach (System.Data.DataRow row in Tab3.Rows){
    //row.
    //*Console.WriteLine(row.Summary());
    double aggThre = (double)row.ItemArray[Tab2.Columns.IndexOf("AgglomerationThreshold")];
    var xValue = (List<double>)row.ItemArray[Tab2.Columns.IndexOf(xVarName)];
    var yValue = (List<double>)row.ItemArray[Tab2.Columns.IndexOf(yVarName)];
    Plot2Ddata.XYvalues xyGroup = new Plot2Ddata.XYvalues("AggThreshold="+aggThre, xValue, yValue);
    plot.dataGroups[i] = xyGroup;
    //ArrayTools.AddToArray(xyGroup, ref plot.dataGroups);
            var dataGroup = plot.dataGroups[i];
            dataGroup.Format.LineColor = allColors[i % allColors.Length];
            dataGroup.Format.PointType = myPointTypes[i % myPointTypes.Length];
            i++;
}

In [ ]:
plot.PlotNow()

In [ ]:
plot.SaveToGIF(yVarName + "_k" + dgDegree + "_2D.png");
plot.SavePgfplotsFile(yVarName + "_k" + dgDegree + "_2D.tex");